In [2]:
#!/usr/bin/env python3
"""
Four-way APSP benchmark: Floyd-Warshall vs Johnson-Dijkstra vs Johnson-Duan
vs Looped Duan, with correctness verification of every solver on every graph.

Self-contained: no imports beyond numpy. Just run it.

--------------------------------------------------------------------------
WHAT IS BEING COMPARED
--------------------------------------------------------------------------
  Floyd-Warshall     O(n^3) dynamic programming. Vectorized over the inner
                     two loops, so its runtime reflects its complexity
                     rather than Python loop overhead.
  Johnson-Dijkstra   Johnson reweighting + Dijkstra once per source.
  Johnson-Duan       Johnson reweighting + BMSSP (Duan, Mao, Mao, Shu, Yin,
                     STOC 2025) once per source.
  Looped Duan        BMSSP once per source, NO Johnson wrapper. This is a
                     separately optimized implementation of the same
                     algorithm (deque-based prepend side, index-based
                     retirement on the insert side, cached block cleaning,
                     and one reusable dense-graph index shared across all
                     n sources).

On graphs with strictly positive weights -- which is what
generate_random_graph produces -- Johnson's potentials are all zero and the
reweighting is the identity. Johnson-Duan and Looped Duan are therefore
solving the same problem with the same algorithm, and the gap between them
measures implementation quality only. Their outputs should agree entry for
entry; the script checks that explicitly and reports it.

Looped Duan requires non-negative weights. It is not a drop-in replacement
for Johnson-Duan on graphs that have negative edges -- handling those is
exactly what the reweighting step buys.

--------------------------------------------------------------------------
RUNTIME WARNING -- READ BEFORE STARTING A FULL n=1500 SWEEP
--------------------------------------------------------------------------
Measured at n=1500, per density, the two Duan variants take roughly
6 to 17 minutes each, growing with density. Across five densities the full
sweep is on the order of TWO HOURS. Floyd-Warshall (~5 s) and
Johnson-Dijkstra (~20-170 s) are negligible next to that.

The script therefore:
  * prints each density's results as soon as they are finished, so an
    interrupted run still gives you everything completed up to that point;
  * writes incremental results to a CSV after every density (--csv);
  * lets you skip solvers (--skip-duan, --skip-looped) or pick densities
    (--densities) so you can split the work across sessions or machines;
  * supports --quick for a fast n=400 sanity run before committing hours.

The five densities are independent. If you have cores, running five
processes with --densities set to one value each turns the wall time into
roughly that of the slowest single density.

--------------------------------------------------------------------------
USAGE
--------------------------------------------------------------------------
    pip install numpy
    python apsp_fourway_benchmark.py --quick              # n=400, ~2 min
    python apsp_fourway_benchmark.py                      # n=1500, ~2 hours
    python apsp_fourway_benchmark.py --densities 0.02,0.05
    python apsp_fourway_benchmark.py --skip-duan          # baselines only
    python apsp_fourway_benchmark.py --csv results.csv

From a Jupyter / Colab / Kaggle cell, either shell out:
    !python apsp_fourway_benchmark.py --quick
or call it in-process with an explicit argument list:
    import apsp_fourway_benchmark as B
    B.main(["--quick"])
Calling B.main() with no arguments from inside a kernel is also safe: the
kernel's own launch flags (-f /root/.../kernel-xxxx.json) are ignored
rather than raising "unrecognized arguments".
"""

import sys
import time
import math
import heapq
import bisect
import argparse
from collections import defaultdict, deque

import numpy as np

sys.setrecursionlimit(20000)
INF = float("inf")


# =====================================================================
# Graph generation, ground truth, and baselines
# =====================================================================
def generate_random_graph(n, density=0.5, max_weight=100):
    """Generate random weighted directed graph (vectorized; identical
    distribution to the interpreted double-loop version)."""
    graph = np.full((n, n), np.inf)
    mask = np.random.random((n, n)) < density
    np.fill_diagonal(mask, False)
    weights = np.random.randint(1, max_weight + 1, size=(n, n)).astype(float)
    graph[mask] = weights[mask]
    np.fill_diagonal(graph, 0)
    return graph


def floyd_warshall_fast(graph):
    """Vectorized ground truth."""
    n = len(graph)
    dist = np.array(graph, dtype=float)
    for k in range(n):
        np.minimum(dist, dist[:, k:k + 1] + dist[k:k + 1, :], out=dist)
    return dist


class AdjacencyList:
    def __init__(self, graph):
        self.n = len(graph)
        graph = np.asarray(graph)
        finite = np.isfinite(graph)
        np.fill_diagonal(finite, False)
        rows, cols = np.nonzero(finite)
        weights = graph[rows, cols]
        self.adj = [[] for _ in range(self.n)]
        for u, v, w in zip(rows.tolist(), cols.tolist(), weights.tolist()):
            self.adj[u].append((v, w))

    def out_edges(self, u):
        return self.adj[u]


def dijkstra(graph, src, n, adj_list=None):
    """Dijkstra's algorithm for single source shortest path."""
    dist = np.full(n, np.inf)
    dist[src] = 0
    pq = [(0, src)]
    visited = set()
    if adj_list is not None:
        while pq:
            d, u = heapq.heappop(pq)
            if u in visited:
                continue
            visited.add(u)
            for v, w in adj_list.out_edges(u):
                new_dist = d + w
                if new_dist < dist[v]:
                    dist[v] = new_dist
                    heapq.heappush(pq, (new_dist, v))
        return dist
    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        for v in range(n):
            if graph[u][v] != np.inf:
                new_dist = dist[u] + graph[u][v]
                if new_dist < dist[v]:
                    dist[v] = new_dist
                    heapq.heappush(pq, (new_dist, v))
    return dist


def _compute_potentials(adj_list, n):
    """Johnson's h-potentials via a supersource with 0-weight edges to every
    vertex. Returns (h, ok); ok=False signals a negative cycle.

    Fast path: with a supersource reaching everything at distance 0 and no
    negative edge anywhere, h is identically 0, so the whole pass is
    skippable. That is exactly the case for generate_random_graph's output
    (weights are randint(1, max_weight)), where the dense version was
    previously one of the largest single costs in the run."""
    has_negative = any(w < 0 for u in range(n) for _, w in adj_list.out_edges(u))
    if not has_negative:
        return [0.0] * n, True

    h = [0.0] * n  # supersource gives every vertex an initial 0
    for _ in range(n):
        changed = False
        for u in range(n):
            hu = h[u]
            for v, w in adj_list.out_edges(u):
                nd = hu + w
                if nd < h[v] - 1e-12:
                    h[v] = nd
                    changed = True
        if not changed:
            return h, True
    for u in range(n):
        hu = h[u]
        for v, w in adj_list.out_edges(u):
            if hu + w < h[v] - 1e-9:
                return h, False
    return h, True


def johnsons_algorithm(graph):
    """Johnson's Algorithm for APSP using standard Dijkstra (control)."""
    n = len(graph)
    adj_in = AdjacencyList(graph)
    h, ok = _compute_potentials(adj_in, n)
    if not ok:
        raise ValueError("Graph contains a negative weight cycle")
    h = np.asarray(h, dtype=float)

    reweighted = np.array(graph, dtype=float)
    finite = np.isfinite(reweighted)
    reweighted[finite] = (reweighted + h[:, None] - h[None, :])[finite]

    dist = np.full((n, n), np.inf)
    adj_list = AdjacencyList(reweighted)
    for u in range(n):
        dist[u] = dijkstra(reweighted, u, n, adj_list=adj_list)
    fin = np.isfinite(dist)
    dist[fin] = (dist - h[:, None] + h[None, :])[fin]
    return dist


# ---------------------------------------------------------------------
# Shared recursion parameters.
#
# The two implementations below were developed separately and originally
# rounded these differently (round() vs int()), which silently gave them
# different recursion depths at some n -- e.g. t=5 vs t=4 at n=1500. That
# would have made the Johnson-Duan vs Looped Duan comparison measure
# "different parameters" as well as "different code". Both now call this,
# so the only difference between them is implementation.
# ---------------------------------------------------------------------
def bmssp_params(n):
    log_n = max(2.0, math.log2(n))
    k = max(2, int(round(log_n ** (1.0 / 3.0))))
    t = max(1, int(round(log_n ** (2.0 / 3.0))))
    level = max(1, int(math.ceil(log_n / t)))
    return k, t, level



# =====================================================================
# Johnson-Duan line: block structure D, BMSSP, and the Johnson wrapper
# =====================================================================
class BlockDS:
    """Two block sequences. D0 holds batch-prepended blocks; D1 holds
    inserted blocks ordered by upper bound and split when they exceed 2M.
    Entries are (value, key) tuples; `best[key]` is the live value and any
    entry disagreeing with it is stale (lazy deletion).

    Note on pull(): merging does NOT stop as soon as M elements have been
    collected. Blocks are pulled whole, so after taking a block from one
    side the other side's front block can still contain something smaller
    than an element already collected. Merging continues until the smallest
    remaining candidate on either side is >= the current M-th smallest
    collected value."""

    __slots__ = ("M", "B", "D0", "D1_bounds", "D1_blocks", "d1_head", "best")

    def __init__(self, M=1, B=math.inf):
        self.M = M if M > 0 else 1
        self.B = B
        self.D0 = deque()
        self.D1_bounds = [B]
        self.D1_blocks = [[]]
        self.d1_head = 0
        self.best = {}

    def _split_smallest(self, items, k):
        if k <= 0:
            return [], items
        if k >= len(items):
            return items, []
        items.sort()
        return items[:k], items[k:]

    def _place_in_D1(self, value, key):
        bounds = self.D1_bounds
        i = bisect.bisect_left(bounds, value, self.d1_head)
        if i == len(bounds):
            i -= 1
        block = self.D1_blocks[i]
        block.append((value, key))
        if len(block) > 2 * self.M:
            small, big = self._split_smallest(block, self.M)
            old_bound = bounds[i]
            self.D1_blocks[i] = small
            bounds[i] = small[-1][0] if small else old_bound
            self.D1_blocks.insert(i + 1, big)
            bounds.insert(i + 1, old_bound)

    def insert(self, key, value):
        best = self.best
        cur = best.get(key)
        if cur is not None and cur <= value:
            return
        best[key] = value
        self._place_in_D1(value, key)

    def batch_prepend(self, items):
        best = self.best
        fresh = []
        for k, v in items:
            cur = best.get(k)
            if cur is None or cur > v:
                best[k] = v
                fresh.append((v, k))
        if not fresh:
            return
        M = self.M
        if len(fresh) <= M:
            self.D0.appendleft(fresh)
            return
        fresh.sort()
        blocks = [fresh[i:i + M] for i in range(0, len(fresh), M)]
        for b in reversed(blocks):
            self.D0.appendleft(b)

    def _clean(self, block):
        best = self.best
        return [vk for vk in block if best.get(vk[1]) == vk[0]]

    def _front_D0(self):
        D0 = self.D0
        while D0:
            blk = self._clean(D0[0])
            if blk:
                return blk
            D0.popleft()
        return []

    def _front_D1(self):
        blocks = self.D1_blocks
        nb = len(blocks)
        while self.d1_head < nb - 1:
            blk = self._clean(blocks[self.d1_head])
            if blk:
                return blk
            self.d1_head += 1
        return self._clean(blocks[nb - 1])

    def is_empty(self):
        return not self._front_D0() and not self._front_D1()

    def pull(self, M=None, B=None):
        """Returns (keys, boundary). M/B accepted for signature
        compatibility with the old heap version; the block size is fixed at
        construction."""
        M = self.M
        collected = []
        threshold = None
        while True:
            d0 = self._front_D0()
            d1 = self._front_D1()
            d0_min = d0[0][0] if len(d0) == 1 else (min(d0)[0] if d0 else None)
            d1_min = d1[0][0] if len(d1) == 1 else (min(d1)[0] if d1 else None)
            if d0_min is None and d1_min is None:
                break
            if len(collected) >= M:
                if d0_min is None:
                    nxt = d1_min
                elif d1_min is None:
                    nxt = d0_min
                else:
                    nxt = d0_min if d0_min < d1_min else d1_min
                if threshold is None:
                    threshold = heapq.nsmallest(M, collected)[-1][0]
                if nxt > threshold:
                    break
            if d1_min is None or (d0_min is not None and d0_min <= d1_min):
                self.D0.popleft()
                collected.extend(d0)
            else:
                if self.d1_head < len(self.D1_blocks) - 1:
                    self.d1_head += 1
                else:
                    self.D1_blocks[-1] = []
                collected.extend(d1)
            threshold = None

        if len(collected) <= M:
            S = collected
        else:
            S, rest = self._split_smallest(collected, M)
            for value, key in rest:
                self._place_in_D1(value, key)

        best = self.best
        for value, key in S:
            best.pop(key, None)

        d0 = self._front_D0()
        d1 = self._front_D1()
        if not d0 and not d1:
            x = self.B
        elif not d0:
            x = min(d1)[0]
        elif not d1:
            x = min(d0)[0]
        else:
            a, b = min(d0)[0], min(d1)[0]
            x = a if a < b else b
        return [key for _, key in S], x


def find_pivots(B, S, d_hat, adj_list, k):
    """k rounds of bounded Bellman-Ford relaxation from S.

    BUG D FIX: a vertex is added to the next frontier whenever its distance
    improves, not only when it is seen for the first time. Gating on
    `v not in W` meant an already-seen vertex that later got a *better*
    distance was never re-expanded from that better value, silently
    freezing stale distances into the pivot search."""
    W = set(S)
    root = {x: x for x in S}
    frontier = set(S)
    limit = k * max(len(S), 1)
    for _ in range(k):
        next_frontier = set()
        for u in frontier:
            du = d_hat[u]
            ru = root.get(u, u)
            for v, w_uv in adj_list.out_edges(u):
                nd = du + w_uv
                if nd >= B:
                    continue
                if nd < d_hat[v]:
                    # strict improvement: must re-expand from the better value
                    d_hat[v] = nd
                    next_frontier.add(v)
                    W.add(v)
                    root[v] = ru
                elif nd == d_hat[v] and v not in W:
                    # equal distance, first sighting: expand once
                    next_frontier.add(v)
                    W.add(v)
                    root[v] = ru
        if not next_frontier:
            break
        frontier = next_frontier
        if len(W) > limit:
            return list(S), W

    counts = defaultdict(int)
    for v in W:
        r = root.get(v)
        if r is not None:
            counts[r] += 1
    P = [x for x in S if counts[x] >= k]
    if not P:
        P = list(S)
    return P, W


def base_case(B, S, d_hat, adj_list, k):
    """BUG E FIX: expand from the minimum-distance vertex of S, not an
    arbitrary set element -- the returned bound is only meaningful relative
    to the smallest starting distance.
    BUG A FIX: return all of U0. The old `{v for v in U0 if d_hat[v] <
    B_prime}` dropped the boundary vertex, after which nothing ever
    expanded its out-edges and anything reachable only through it was
    lost."""
    if not S:
        return B, set()
    x = min(S, key=lambda v: d_hat[v])
    U0 = set()
    visited = set()
    pq = [(d_hat[x], x)]
    while pq and len(U0) < k + 1:
        du, u = heapq.heappop(pq)
        if u in visited or du > d_hat[u] + 1e-12 or du >= B:
            continue
        visited.add(u)
        U0.add(u)
        duu = d_hat[u]
        for v, w_uv in adj_list.out_edges(u):
            nd = duu + w_uv
            if nd >= B:
                continue
            if nd < d_hat[v]:
                d_hat[v] = nd
                heapq.heappush(pq, (nd, v))
            elif nd == d_hat[v] and v not in visited:
                # already at the correct distance but not yet expanded:
                # it still needs to be popped so its out-edges are explored
                heapq.heappush(pq, (nd, v))
    if len(U0) <= k:
        return B, U0
    return max(d_hat[v] for v in U0), U0


def bmssp(level, B, S, d_hat, adj_list, k, t):
    """BUG C, REVISED: the paper's size budget is kept, not deleted.

    An earlier revision removed the `len(U) < target` exit outright and
    drained D fully. That is correct but slow -- profiling showed ~1,450
    recursive calls per source at n=300, because at level 1 (M=1) it pulls
    and completes one vertex at a time across the whole graph. The budget
    exists precisely to stop that.

    The budget is only safe when the unfinished work is handed back, and
    the two halves must agree:
      - on a budget-triggered exit, return B' = the last pulled boundary,
        and report as complete only the W-vertices below it;
      - on a natural exit (D drained), return B' = B and all of W.
    Vertices that were pulled but not completed by the child are pushed
    back into D via batch_prepend, so the parent's loop re-pulls them.
    This is the hand-back the original code already had; it could not work
    while bugs A/D/E were corrupting what the recursion returned."""
    if level == 0:
        return base_case(B, S, d_hat, adj_list, k)
    if not S:
        return B, set()

    P, W = find_pivots(B, S, d_hat, adj_list, k)

    M = 1 << ((level - 1) * t)
    D = BlockDS(M, B)
    for x in P:
        if d_hat[x] < B:
            D.insert(x, d_hat[x])

    U = set()
    B_prime = min((d_hat[x] for x in P), default=B)
    target = k * (1 << (level * t))
    budget_hit = False
    guard = 0
    guard_limit = max(50000, 50 * adj_list.n)

    while guard < guard_limit:
        if len(U) >= target:
            budget_hit = True
            break
        S_i, B_i = D.pull()
        if not S_i:
            break
        guard += 1
        B_prime_i, U_i = bmssp(level - 1, B_i, set(S_i), d_hat, adj_list, k, t)
        U |= U_i
        B_prime = B_prime_i

        K = []
        for u in U_i:
            du = d_hat[u]
            for v, w_uv in adj_list.out_edges(u):
                nd = du + w_uv
                # `<=`, not `<`: a vertex already at its correct distance
                # must STILL re-enter D so it gets pulled and its out-edges
                # explored (the original Fix #1, and load-bearing). An
                # attempt to suppress tie-reinsertion for
                # already-expanded vertices was tried here and measured
                # ~2x faster but broke exactness on 7 of 16 configs
                # (max_ratio up to 2.13), so it was reverted: the
                # suppression is not sound as stated.
                if nd <= d_hat[v]:
                    d_hat[v] = nd
                    if nd < B:
                        if nd >= B_i:
                            D.insert(v, nd)
                        elif nd >= B_prime_i:
                            K.append((v, nd))
                        else:
                            D.insert(v, nd)
        # hand back pulled-but-unfinished vertices so the parent re-pulls them
        for x in S_i:
            dx = d_hat[x]
            if B_prime_i <= dx < B_i:
                K.append((x, dx))
        if K:
            D.batch_prepend(K)

    if budget_hit:
        final_B = B_prime if B_prime < B else B
        U |= {x for x in W if d_hat[x] < final_B}
        return final_B, U
    U |= W
    return B, U


def sssp_bmssp(graph, src, cleanup_rounds=0, adj_list=None, n=None):
    """Single-source BMSSP. `cleanup_rounds` defaults to 0: with bugs A-E
    fixed the result is already exact, and the sweeps were an O(n + m)
    per-round, per-source tax that dominated the algorithm being measured.
    Non-zero values are retained only for demo_deliberate_approximation."""
    if adj_list is None:
        adj_list = AdjacencyList(graph)
    if n is None:
        n = adj_list.n

    d_hat = [math.inf] * n          # plain list: scalar numpy indexing is
    d_hat[src] = 0.0                # several times slower in these loops

    k, t, level = bmssp_params(n)

    bmssp(level, math.inf, {src}, d_hat, adj_list, k, t)

    rounds_used = 0
    for _ in range(cleanup_rounds):
        changed = False
        for u in range(n):
            du = d_hat[u]
            if du == math.inf:
                continue
            for v, w_uv in adj_list.out_edges(u):
                nd = du + w_uv
                if nd < d_hat[v] - 1e-12:
                    d_hat[v] = nd
                    changed = True
        rounds_used += 1
        if not changed:
            break
    sssp_bmssp.last_rounds_used = rounds_used
    return d_hat


def johnsons_algorithm_bmssp_fixed(graph, cleanup_rounds=0):
    """Johnson's reweighting + BMSSP per source. Reweighting and
    un-reweighting are vectorized; the h-pass skips entirely when no edge
    is negative."""
    n = len(graph)
    adj_in = AdjacencyList(graph)
    h, ok = _compute_potentials(adj_in, n)
    if not ok:
        raise ValueError("Graph contains a negative weight cycle")
    h = np.asarray(h, dtype=float)

    reweighted = np.array(graph, dtype=float)
    finite = np.isfinite(reweighted)
    reweighted[finite] = (reweighted + h[:, None] - h[None, :])[finite]

    adj_list = AdjacencyList(reweighted)
    dist = np.empty((n, n), dtype=float)
    for u in range(n):
        dist[u] = sssp_bmssp(reweighted, u, cleanup_rounds=cleanup_rounds,
                             adj_list=adj_list, n=n)

    fin = np.isfinite(dist)
    dist[fin] = (dist - h[:, None] + h[None, :])[fin]
    return dist


# =====================================================================
# Looped Duan line: the same algorithm, separately optimized.
# Renamed BlockLinkedListDFast -> BlockDSFast to avoid clashing with the
# BlockDS above; the two are independent implementations and are compared
# against each other on purpose.
# =====================================================================
class BlockDSFast:
    """Entries are (value, key) tuples. `best[key]` is the current live
    value for key; an entry whose value != best[key] is stale (lazy
    deletion), exactly as in the reference implementation."""

    __slots__ = ("M", "B", "D0", "D1_bounds", "D1_blocks", "d1_head", "best")

    def __init__(self, M, B):
        self.M = M if M > 0 else 1
        self.B = B
        self.D0 = deque()
        self.D1_bounds = [B]
        self.D1_blocks = [[]]
        self.d1_head = 0
        self.best = {}

    # -- selection -------------------------------------------------------
    def _split_smallest(self, items, k):
        """(k smallest, rest). C-speed sort; see note 5 in the module docstring."""
        if k <= 0:
            return [], items
        if k >= len(items):
            return items, []
        items.sort()
        return items[:k], items[k:]

    # -- D1 placement ----------------------------------------------------
    def _place_in_D1(self, value, key):
        bounds = self.D1_bounds
        i = bisect.bisect_left(bounds, value, self.d1_head)
        if i == len(bounds):
            i -= 1
        block = self.D1_blocks[i]
        block.append((value, key))
        if len(block) > 2 * self.M:
            small, big = self._split_smallest(block, self.M)
            old_bound = bounds[i]
            self.D1_blocks[i] = small
            bounds[i] = small[-1][0] if small else old_bound
            self.D1_blocks.insert(i + 1, big)
            bounds.insert(i + 1, old_bound)

    # -- public ops ------------------------------------------------------
    def insert(self, key, value):
        best = self.best
        cur = best.get(key)
        if cur is not None and cur <= value:
            return
        best[key] = value
        self._place_in_D1(value, key)

    def batch_prepend(self, items):
        """items: iterable of (key, value), all smaller than everything
        currently in D (the caller's [B'_i, B_i) discipline guarantees it)."""
        best = self.best
        fresh = []
        for k, v in items:
            cur = best.get(k)
            if cur is None or cur > v:
                best[k] = v
                fresh.append((v, k))
        if not fresh:
            return
        M = self.M
        if len(fresh) <= M:
            self.D0.appendleft(fresh)
            return
        fresh.sort()
        blocks = [fresh[i:i + M] for i in range(0, len(fresh), M)]
        for b in reversed(blocks):
            self.D0.appendleft(b)

    # -- internals -------------------------------------------------------
    def _clean(self, block):
        best = self.best
        return [vk for vk in block if best.get(vk[1]) == vk[0]]

    def _front_D0(self):
        """Cleaned front block of D0 (or []), retiring dead blocks."""
        D0 = self.D0
        while D0:
            blk = self._clean(D0[0])
            if blk:
                return blk
            D0.popleft()
        return []

    def _front_D1(self):
        """Cleaned front block of D1 (or []), advancing d1_head past dead ones."""
        blocks = self.D1_blocks
        n = len(blocks)
        while self.d1_head < n - 1:
            blk = self._clean(blocks[self.d1_head])
            if blk:
                return blk
            self.d1_head += 1
        return self._clean(blocks[n - 1])

    def is_empty(self):
        return not self._front_D0() and not self._front_D1()

    def pull(self):
        """Return (S, x): S is up to M (key, value) pairs with the smallest
        values; x separates S from everything remaining (x = B if empty).

        Merging continues past len(collected) >= M until the smallest
        remaining candidate on either side is >= the current M-th smallest
        collected value: blocks are pulled whole, so reaching M elements is
        not by itself a safe stopping point."""
        M = self.M
        collected = []
        threshold = None
        while True:
            d0 = self._front_D0()
            d1 = self._front_D1()
            d0_min = d0[0][0] if len(d0) == 1 else (min(d0)[0] if d0 else None)
            d1_min = d1[0][0] if len(d1) == 1 else (min(d1)[0] if d1 else None)

            if d0_min is None and d1_min is None:
                break

            if len(collected) >= M:
                if d0_min is None:
                    nxt = d1_min
                elif d1_min is None:
                    nxt = d0_min
                else:
                    nxt = d0_min if d0_min < d1_min else d1_min
                if threshold is None:
                    threshold = heapq.nsmallest(M, collected)[-1][0]
                if nxt > threshold:
                    break

            if d1_min is None or (d0_min is not None and d0_min <= d1_min):
                self.D0.popleft()
                collected.extend(d0)
            else:
                if self.d1_head < len(self.D1_blocks) - 1:
                    self.d1_head += 1
                else:
                    self.D1_blocks[-1] = []
                collected.extend(d1)
            threshold = None  # collected changed; recompute lazily if needed

        if len(collected) <= M:
            S = collected
        else:
            S, rest = self._split_smallest(collected, M)
            for value, key in rest:
                self._place_in_D1(value, key)

        best = self.best
        for value, key in S:
            best.pop(key, None)

        d0 = self._front_D0()
        d1 = self._front_D1()
        if not d0 and not d1:
            x = self.B
        elif not d0:
            x = min(d1)[0]
        elif not d1:
            x = min(d0)[0]
        else:
            a, b = min(d0)[0], min(d1)[0]
            x = a if a < b else b
        return [(key, value) for value, key in S], x


def prepare_graph(graph):
    """Build the dense-index representation ONCE so it can be reused across
    many sources. Returns (nodes, idx, adj) where adj is indexed 0..n-1.

    This matters a lot for APSP: remapping is O(n + m), and at n=1500 /
    p=0.2 (450k edges) profiling showed the per-call remap dominating the
    entire run - it was more expensive than the shortest-path computation
    itself. Paying it once instead of n times is the single biggest win for
    looped use."""
    nodes = list(graph.keys())
    idx = {u: i for i, u in enumerate(nodes)}
    adj = [None] * len(nodes)
    for u, i in idx.items():
        adj[i] = [(idx[v], w) for v, w in graph[u]]
    return nodes, idx, adj


def duan2025_sssp_prepared(prep, src_index):
    """Core solver on an already-prepared dense graph. Returns a plain list
    `dist` indexed 0..n-1 (no dict rebuild), so callers that loop over many
    sources never pay dict-construction cost either."""
    nodes, idx, adj = prep
    n = len(nodes)
    dist = [INF] * n
    dist[src_index] = 0.0
    if n <= 4:
        # tiny graph: plain Dijkstra on the dense representation
        pq = [(0.0, src_index)]
        visited = set()
        while pq:
            d, u = heapq.heappop(pq)
            if u in visited:
                continue
            visited.add(u)
            for v, w in adj[u]:
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
        return dist

    k, t, L = bmssp_params(n)

    def find_pivots(B, S):
        W = set(S)
        W_prev = S
        root = {x: x for x in S}
        limit = k * (len(S) if S else 1)
        for _ in range(k):
            W_next = set()
            for u in W_prev:
                du = dist[u]
                ru = root.get(u, u)
                for v, w in adj[u]:
                    nd = du + w
                    if nd <= dist[v] and nd < B:
                        dist[v] = nd
                        W_next.add(v)
                        root[v] = ru
            if len(W) + len(W_next) > limit:
                return set(S), W | W_next
            W |= W_next
            W_prev = W_next
            if not W_prev:
                break
        counts = defaultdict(int)
        for x in W:
            counts[root.get(x, x)] += 1
        P = {x for x in S if counts.get(x, 1) >= k}
        if not P and S:
            P = {min(S, key=lambda x: dist[x])}
        return P, W

    def base_case(B, S):
        x = min(S, key=lambda v: dist[v])
        heap = [(dist[x], x)]
        visited = set()
        U0 = set()
        kp1 = k + 1
        while heap and len(U0) < kp1:
            d, u = heapq.heappop(heap)
            if u in visited or d > dist[u] + 1e-12 or d >= B:
                continue
            visited.add(u)
            U0.add(u)
            du = dist[u]
            for v, w in adj[u]:
                nd = du + w
                if nd < dist[v] and nd < B:
                    dist[v] = nd
                    heapq.heappush(heap, (nd, v))
        if len(U0) <= k:
            return B, U0
        return max(dist[u] for u in U0), U0

    def bmssp(l, B, S):
        if l == 0:
            return base_case(B, S)
        P, W = find_pivots(B, S)
        D = BlockDSFast(1 << ((l - 1) * t), B)
        for p in P:
            if dist[p] < B:
                D.insert(p, dist[p])
        B_prime = min((dist[p] for p in P), default=B)
        U = set()
        guard = 0
        guard_limit = 50 * n if 50 * n > 50000 else 50000
        while guard < guard_limit:
            S_kv, Bi = D.pull()
            if not S_kv:
                break
            guard += 1
            Si = {key for key, _ in S_kv}
            Bi_prime, Ui = bmssp(l - 1, Bi, Si)
            U |= Ui
            batch = []
            for u in Ui:
                du = dist[u]
                for v, w in adj[u]:
                    nd = du + w
                    if nd <= dist[v]:
                        dist[v] = nd
                        if nd < B:
                            if nd >= Bi:
                                D.insert(v, nd)
                            elif nd >= Bi_prime:
                                batch.append((v, nd))
                            else:
                                D.insert(v, nd)
            if batch:
                D.batch_prepend(batch)
            B_prime = Bi_prime
        U |= W
        return (B_prime if B_prime < B else B), U

    bmssp(L, INF, {src_index})
    return dist


def looped_duan_apsp(graph, show_progress=False, progress_every=250):
    """BMSSP once per source, no Johnson wrapper. Graph preparation happens
    once and is reused across all n sources."""
    n = len(graph)
    g = np.asarray(graph)
    finite = np.isfinite(g)
    np.fill_diagonal(finite, False)
    rows, cols = np.nonzero(finite)
    adj_dict = {u: [] for u in range(n)}
    for u, v, w in zip(rows.tolist(), cols.tolist(), g[rows, cols].tolist()):
        adj_dict[u].append((v, w))

    prep = prepare_graph(adj_dict)
    out = np.empty((n, n), dtype=float)
    for i in range(n):
        out[i] = duan2025_sssp_prepared(prep, i)
        if show_progress and ((i + 1) % progress_every == 0 or i + 1 == n):
            print(f"      looped Duan: {i + 1}/{n} sources", flush=True)
    return out


# =====================================================================
# Verification
# =====================================================================

def verify(exact, approx, tol=1e-7):
    """Compare a distance matrix against ground truth, separating the
    failure modes. A single pass/fail flag hides which kind of wrong you
    have: an overestimate, a missed path and an invented path are three
    different bugs.
        missed     -- reachable in truth, reported unreachable
        phantom    -- unreachable in truth, reported reachable
        undershoot -- distance BELOW the true shortest path (impossible for
                      a sound method; a nonzero count means a real bug)
    """
    n = exact.shape[0]
    off = ~np.eye(n, dtype=bool)
    ef = np.isfinite(exact) & off
    af = np.isfinite(approx) & off
    valid = ef & af & (exact > tol)
    nv = int(valid.sum())
    return {
        "pct_exact": 100.0 * np.sum(np.abs(approx[valid] - exact[valid])
                                    <= tol * np.maximum(1.0, exact[valid])) / nv if nv else float("nan"),
        "max_ratio": float(np.max(approx[valid] / exact[valid])) if nv else float("nan"),
        "missed": int(np.sum(ef & ~af)),
        "phantom": int(np.sum(~ef & af)),
        "undershoot": int(np.sum(valid & (approx < exact - tol))),
        "n_pairs": nv,
    }


def is_exact(v):
    return (v["pct_exact"] == 100.0 and v["missed"] == 0
            and v["phantom"] == 0 and v["undershoot"] == 0)


# =====================================================================
# Benchmark
# =====================================================================

def run(n, densities, seed=42, skip_duan=False, skip_looped=False,
        csv_path=None, progress=True):
    np.random.seed(seed)
    rows = []

    if csv_path:
        with open(csv_path, "w") as f:
            f.write("n,density,edges,t_fw,t_johnson_dijkstra,t_johnson_duan,"
                    "t_looped_duan,exact_dij,exact_jduan,exact_lduan,jd_vs_ld_diffs\n")

    for d in densities:
        graph = generate_random_graph(n, density=d)
        m = int(np.sum(np.isfinite(graph) & (graph != 0)))
        print("=" * 96)
        print(f"n={n}  density={d:.2%}  edges={m:,}")
        print("=" * 96, flush=True)

        t0 = time.time(); exact = floyd_warshall_fast(graph); t_fw = time.time() - t0
        print(f"  Floyd-Warshall     {t_fw:8.2f}s   (ground truth)", flush=True)

        t0 = time.time(); r_dij = johnsons_algorithm(graph); t_dij = time.time() - t0
        v_dij = verify(exact, r_dij)
        print(f"  Johnson-Dijkstra   {t_dij:8.2f}s   exact={v_dij['pct_exact']:6.2f}%  "
              f"missed={v_dij['missed']} phantom={v_dij['phantom']} under={v_dij['undershoot']}",
              flush=True)

        t_jd = float("nan"); v_jd = None; r_jd = None
        if not skip_duan:
            print(f"  Johnson-Duan       running ...", flush=True)
            t0 = time.time()
            r_jd = johnsons_algorithm_bmssp_fixed(graph, cleanup_rounds=0)
            t_jd = time.time() - t0
            v_jd = verify(exact, r_jd)
            print(f"  Johnson-Duan       {t_jd:8.2f}s   exact={v_jd['pct_exact']:6.2f}%  "
                  f"missed={v_jd['missed']} phantom={v_jd['phantom']} under={v_jd['undershoot']}",
                  flush=True)

        t_ld = float("nan"); v_ld = None; r_ld = None
        if not skip_looped:
            print(f"  Looped Duan        running ...", flush=True)
            t0 = time.time()
            r_ld = looped_duan_apsp(graph, show_progress=progress,
                                    progress_every=max(1, n // 6))
            t_ld = time.time() - t0
            v_ld = verify(exact, r_ld)
            print(f"  Looped Duan        {t_ld:8.2f}s   exact={v_ld['pct_exact']:6.2f}%  "
                  f"missed={v_ld['missed']} phantom={v_ld['phantom']} under={v_ld['undershoot']}",
                  flush=True)

        diffs = ""
        if r_jd is not None and r_ld is not None:
            nd = int(np.sum(np.abs(r_jd - r_ld) > 1e-7))
            diffs = nd
            print(f"  Johnson-Duan vs Looped Duan: {nd} differing entries "
                  f"({'identical' if nd == 0 else 'MISMATCH'})", flush=True)

        # ratios only where both sides verified exact
        def ratio(t, v):
            return (t / t_dij) if (v is not None and is_exact(v) and is_exact(v_dij)) else None
        r1, r2 = ratio(t_jd, v_jd), ratio(t_ld, v_ld)
        if r1: print(f"  -> Johnson-Duan is {r1:.1f}x slower than Johnson-Dijkstra")
        if r2: print(f"  -> Looped Duan  is {r2:.1f}x slower than Johnson-Dijkstra")
        if r1 and r2:
            rr = t_jd / t_ld
            print(f"  -> Looped Duan is {rr:.2f}x faster than Johnson-Duan" if rr >= 1
                  else f"  -> Looped Duan is {1/rr:.2f}x SLOWER than Johnson-Duan")
        print(flush=True)

        rows.append(dict(n=n, d=d, m=m, t_fw=t_fw, t_dij=t_dij, t_jd=t_jd, t_ld=t_ld,
                         ok_dij=is_exact(v_dij),
                         ok_jd=is_exact(v_jd) if v_jd else None,
                         ok_ld=is_exact(v_ld) if v_ld else None,
                         diffs=diffs))

        if csv_path:  # append after every density so an interrupted run keeps its data
            with open(csv_path, "a") as f:
                r = rows[-1]
                f.write(f"{n},{d},{m},{t_fw:.4f},{t_dij:.4f},{t_jd:.4f},{t_ld:.4f},"
                        f"{r['ok_dij']},{r['ok_jd']},{r['ok_ld']},{diffs}\n")

    # ---- summary ----
    print("=" * 96)
    print(f"SUMMARY  (n={n})")
    print("=" * 96)
    print(f"{'density':>8} {'edges':>9} {'FW(s)':>8} {'J-Dij(s)':>9} {'J-Duan(s)':>10} "
          f"{'Loop(s)':>9} {'JD/Dij':>8} {'LD/Dij':>8} {'JD/LD':>7} {'all exact':>10}")
    for r in rows:
        oks = [r['ok_dij'], r['ok_jd'], r['ok_ld']]
        allok = all(o for o in oks if o is not None)
        f_jd = f"{r['t_jd']/r['t_dij']:.1f}x" if r['t_jd'] == r['t_jd'] else "-"
        f_ld = f"{r['t_ld']/r['t_dij']:.1f}x" if r['t_ld'] == r['t_ld'] else "-"
        f_r  = (f"{r['t_jd']/r['t_ld']:.2f}x"
                if r['t_jd'] == r['t_jd'] and r['t_ld'] == r['t_ld'] else "-")
        print(f"{r['d']:>8.0%} {r['m']:>9,} {r['t_fw']:>8.2f} {r['t_dij']:>9.2f} "
              f"{r['t_jd']:>10.2f} {r['t_ld']:>9.2f} {f_jd:>8} {f_ld:>8} {f_r:>7} "
              f"{'yes' if allok else 'NO':>10}")

    if len(rows) >= 2:
        lm = np.log([r["m"] for r in rows])
        print("\nedge-scaling exponents (both Johnson bounds are linear in m, so")
        print("theory predicts ~1.0; Floyd-Warshall ignores m, so ~0 is correct):")
        for name, key in [("Johnson-Dijkstra", "t_dij"), ("Johnson-Duan", "t_jd"),
                          ("Looped Duan", "t_ld"), ("Floyd-Warshall", "t_fw")]:
            vals = [r[key] for r in rows]
            if any(v != v for v in vals):
                continue
            print(f"    {name:18s}: {np.polyfit(lm, np.log(vals), 1)[0]:5.2f}")
    if csv_path:
        print(f"\nresults written to {csv_path}")
    return rows


def main(argv=None):
    ap = argparse.ArgumentParser(description="Four-way APSP benchmark")
    ap.add_argument("--n", type=int, default=1500)
    ap.add_argument("--densities", type=str, default="0.02,0.05,0.1,0.2,0.4")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--skip-duan", action="store_true", help="skip Johnson-Duan")
    ap.add_argument("--skip-looped", action="store_true", help="skip Looped Duan")
    ap.add_argument("--csv", type=str, default=None, help="write incremental results here")
    ap.add_argument("--quick", action="store_true", help="n=400 sanity run (~2 min)")
    ap.add_argument("--no-progress", action="store_true")
    # parse_known_args, not parse_args: a running Jupyter/Colab/Kaggle kernel
    # puts its own '-f /root/.../kernel-xxxx.json' in sys.argv, and
    # parse_args would abort on it.
    args, unknown = ap.parse_known_args(argv)
    if unknown and argv is None:
        print(f"(ignoring {len(unknown)} unrecognized argument(s), typically "
              f"notebook-kernel launch flags: {unknown})\n")

    n = 400 if args.quick else args.n
    densities = [float(x) for x in args.densities.split(",")]

    est = len(densities) * (6 if n >= 1500 else 0.3)
    print(f"n={n}, {len(densities)} densities: {densities}")
    if n >= 1500 and not (args.skip_duan and args.skip_looped):
        print(f"NOTE: at n=1500 each Duan variant takes roughly 6-17 min per density.")
        print(f"      Expect on the order of {est:.0f}-{est*3:.0f} minutes total.")
        print(f"      Results print (and append to --csv) after every density.\n")

    run(n, densities, seed=args.seed, skip_duan=args.skip_duan,
        skip_looped=args.skip_looped, csv_path=args.csv,
        progress=not args.no_progress)


if __name__ == "__main__":
    main()

(ignoring 2 unrecognized argument(s), typically notebook-kernel launch flags: ['-f', '/root/.local/share/jupyter/runtime/kernel-15d7eddc-e29a-4d8b-b0c6-ea80dc8192fd.json'])

n=1500, 5 densities: [0.02, 0.05, 0.1, 0.2, 0.4]
NOTE: at n=1500 each Duan variant takes roughly 6-17 min per density.
      Expect on the order of 30-90 minutes total.
      Results print (and append to --csv) after every density.

n=1500  density=2.00%  edges=45,097
  Floyd-Warshall        10.73s   (ground truth)
  Johnson-Dijkstra      24.21s   exact=100.00%  missed=0 phantom=0 under=0
  Johnson-Duan       running ...
  Johnson-Duan         392.46s   exact=100.00%  missed=0 phantom=0 under=0
  Looped Duan        running ...
      looped Duan: 250/1500 sources
      looped Duan: 500/1500 sources
      looped Duan: 750/1500 sources
      looped Duan: 1000/1500 sources
      looped Duan: 1250/1500 sources
      looped Duan: 1500/1500 sources
  Looped Duan          316.18s   exact=100.00%  missed=0 phantom=0 under=0